# 02 · Qualidade dos dados e transformação → camada silver

**MVP — Onde subsidiar o frete no e-commerce brasileiro: pipeline de dados em nuvem (Databricks)**

Universidade de Brasília (UnB) · Departamento de Engenharia de Produção · Sistemas de Suporte à Decisão
Prof. André Luiz Marques Serrano · **Aluna: Brunna Gavião Moraes — 231025341**


A análise de qualidade vem **antes** da transformação, porque é ela que define as regras de tratamento:

1. perfil de **todos os atributos** dos 10 arquivos da bronze;
2. **33 verificações** em sete dimensões (completude, unicidade, conformidade, consistência, acurácia, integridade e
   atualidade), cada uma ligada ao tratamento adotado;
3. regras **R01–R13**, que geram as tabelas da camada silver (Delta) e registram a linhagem.

In [ ]:
%pip install -q kagglehub statsmodels

In [ ]:
# Configuração comum: localiza o repositório, importa o módulo de apoio e detecta a plataforma.
# No Databricks a variável `spark` já existe; fora dele o mesmo código roda localmente (Parquet + DuckDB).
import os, sys, importlib
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
import mvp_frete as mvp
importlib.reload(mvp)

import pandas as pd
pd.set_option("display.max_columns", 50, "display.width", 200)

plat = mvp.Plataforma(spark if "spark" in globals() else None)
print("Ambiente :", plat.preparar())
print("Bronze   :", plat.dir_bronze)
print("Evidências:", plat.dir_evidencias)

## 1. Leitura da bronze e conferência dos hashes

In [ ]:
import json
dir_coleta = sorted(p for p in plat.dir_bronze.iterdir() if p.name.startswith("coleta="))[-1]
meta = json.loads((dir_coleta / "_metadados_coleta.json").read_text(encoding="utf-8"))
conf = pd.DataFrame([{"arquivo": m["arquivo"], "hash_confere": mvp.sha256(dir_coleta / m["arquivo"]) == m["sha256"]}
                     for m in meta["arquivos"]])
assert conf.hash_confere.all(), "Arquivo da bronze alterado depois da coleta!"
print("Coleta usada:", dir_coleta.name, "· hashes conferidos:", conf.hash_confere.sum(), "de", len(conf))

b = mvp.ler_bronze(dir_coleta)   # tudo lido como texto: nenhuma conversão silenciosa
display(pd.DataFrame({t: [len(df), df.shape[1]] for t, df in b.items()}, index=["linhas", "colunas"]).T)

## 2. Perfil de todos os atributos

Para cada atributo: tipo esperado, nulos, valores distintos, % de valores conformes ao tipo, mínimo, máximo e valores
mais frequentes.

In [ ]:
perfil = mvp.perfil_atributos(b)
mvp.salvar_tabela_evidencia(perfil, plat, "q1_perfil_por_atributo")
print(len(perfil), "atributos perfilados ·", (perfil.nulos > 0).sum(), "com nulos ·",
      (perfil.pct_conformidade_tipo < 100).sum(), "com valores fora do tipo")
display(perfil)

## 3. Verificações por dimensão de qualidade

In [ ]:
verif = mvp.verificacoes_qualidade(b)
mvp.salvar_tabela_evidencia(verif, plat, "q2_verificacoes_por_dimensao")
display(verif.groupby("dimensao", sort=False).agg(verificacoes=("verificacao", "size"),
                                                  com_ocorrencia=("ocorrencias", lambda x: (x > 0).sum())))
display(verif)

## 4. Atualidade: volume mensal e janela de análise

In [ ]:
import matplotlib.pyplot as plt
vol = mvp.volume_mensal(b)
mvp.salvar_tabela_evidencia(vol, plat, "q3_volume_mensal")

fig, ax = plt.subplots(figsize=(11, 3.6))
cores = ["#2a78d6" if j else "#c3c2b7" for j in vol.na_janela]
ax.bar(vol.ano_mes, vol.pedidos, color=cores, width=0.8)
ax.set_title("Pedidos por mês de compra — em azul, a janela de análise (jan/2017 a ago/2018)", loc="left", fontsize=11)
ax.set_ylabel("pedidos")
ax.tick_params(axis="x", rotation=90, labelsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.6); ax.set_axisbelow(True)
mvp.salvar_figura(fig, plat, "q3_volume_mensal"); plt.show()

### Leitura da qualidade

* **A base é íntegra nas chaves:** nenhum pedido, item, produto ou vendedor duplicado, e nenhuma chave estrangeira órfã
  entre itens, produtos, vendedores e clientes.
* **Os problemas que afetam o frete são poucos e têm tratamento explícito:**
  * produtos sem categoria (R05);
  * 4 produtos com peso zero, tratados como ausentes, e 2 sem medidas (R06);
  * coordenadas fora do Brasil, descartadas antes do centroide (R04);
  * CEPs sem geolocalização, que deixam a distância nula, sem excluir o item (R10).
* **Valores extremos que são o próprio fenômeno estudado foram mantidos e sinalizados:**
  * frete maior que o preço do item;
  * frete zero (frete grátis).
* **Atualidade:** 2016 tem volume irrisório e set–out/2018 está truncado. A janela vai de **jan/2017 a ago/2018**
  (R08), com 20 meses completos.

## 5. Transformação bronze → silver (R01–R13)

In [ ]:
log = mvp.LogLinhagem()
S = mvp.transformar_silver(b, log)
comentarios = {
    "pedidos": "Pedidos no escopo (jan/2017–ago/2018, exceto cancelados/indisponíveis) — R01, R08",
    "itens": "Itens dos pedidos no escopo, com preço e frete em R$ — R01, R08",
    "clientes": "Clientes: CEP (5 dígitos), cidade e UF; sem identificador único (LGPD) — R02, R03, R13",
    "vendedores": "Vendedores: CEP (5 dígitos), cidade e UF — R02, R03",
    "produtos": "Produtos com categoria, medidas, peso cubado e peso taxado — R05, R06",
    "avaliacoes": "Uma avaliação por pedido, sem texto livre — R07, R13",
    "geo_cep": "Centroide (mediana) de latitude e longitude por prefixo de CEP — R04",
    "ibge_uf": "População estimada 2018 por UF (IBGE), chaveada pela sigla — R12",
}
linhas_silver = []
for tabela, df in S.items():
    n = plat.gravar(df, "silver", tabela, comentarios[tabela])
    linhas_silver.append({"tabela": plat.nome_tabela("silver", tabela), "linhas": n, "colunas": df.shape[1]})
display(pd.DataFrame(linhas_silver))

In [ ]:
log_df = log.df()
plat.gravar(log_df, "silver", "meta_log_transformacoes", "Log de linhagem das regras aplicadas na silver")
display(log_df)

> **Evidência a capturar:** *Catalog Explorer → workspace → frete_silver* com as tabelas criadas
> (`evidencias/02_silver_tabelas.png`).